# Data Cleaning & Preprocessing Notebook

Preprocessing for the longitudinal stylometric drift study.
Files: `data_long.csv` (long format), `data_wide.csv` (wide format).

## 1. Load and inspect data

In [ ]:
import pandas as pd
import numpy as np

long_df = pd.read_csv('data_long.csv')
wide_df = pd.read_csv('data_wide.csv')

print(long_df.info())
print(long_df.head())
print(wide_df.head())
print('Shapes:', long_df.shape, wide_df.shape)

## 2. Missing value handling

In [ ]:
# Count missing values per column
print('Long NaNs:\n', long_df.isna().sum())
print('Wide NaNs:\n', wide_df.isna().sum())

In [ ]:
# Strategy:
# - Numeric scores: impute with group (Profile) median at that time point
# - Drop participants missing an entire wave (cannot model change)
long_df['Stylometry_Composite'] = (
    long_df.groupby(['Profile','TimePoint'])['Stylometry_Composite']
           .transform(lambda s: s.fillna(s.median())))

# Participants with incomplete waves
complete = long_df.groupby('ParticipantID')['TimePoint'].nunique()
drop_ids = complete[complete < long_df['TimePoint'].nunique()].index
long_df = long_df[~long_df['ParticipantID'].isin(drop_ids)]
print('Dropped participants:', list(drop_ids))

# Wide format: drop rows with any missing wave
wide_df = wide_df.dropna(subset=['T1','T2','T3'])
print('Wide shape after cleaning:', wide_df.shape)

## 3. Duplicate & consistency checks

In [ ]:
print('Duplicate long rows:', long_df.duplicated().sum())
print('Duplicate participants:', long_df['ParticipantID'].duplicated().sum())
assert long_df.groupby('ParticipantID')['Profile'].nunique().max() == 1, 'Profile inconsistency'
print('OK: no duplicates, profiles consistent per participant')

## 4. Outlier screening

In [ ]:
z = (long_df['Stylometry_Composite'] - long_df['Stylometry_Composite'].mean()) / long_df['Stylometry_Composite'].std()
outliers = long_df[z.abs() > 3]
print(f'{len(outliers)} observations with |z| > 3')
print(outliers)

## 5. Data transformation

In [ ]:
# Recode categorical factors with ordered levels
long_df['TimePoint'] = pd.Categorical(long_df['TimePoint'], ['T1','T2','T3'], ordered=True)
long_df['Profile'] = long_df['Profile'].astype('category')

# Recompute drift (wide format) and verify T3-T1 consistency
wide_df['Drift_recalc'] = wide_df['T3'] - wide_df['T1']
if 'Drift' in wide_df:
    print('Max |Drift - recalc|:', (wide_df['Drift'] - wide_df['Drift_recalc']).abs().max())

# Standardize composite (z-score) for modeling
long_df['Composite_z'] = (long_df['Stylometry_Composite'] - long_df['Stylometry_Composite'].mean()) / long_df['Stylometry_Composite'].std()

## 6. Save cleaned outputs

In [ ]:
long_df.to_csv('data_long_clean.csv', index=False)
wide_df.to_csv('data_wide_clean.csv', index=False)
print('Saved data_long_clean.csv, data_wide_clean.csv')
print(long_df.describe())